In [2]:
import os
import pandas as pd
import time
from openai import AzureOpenAI, OpenAI
from dotenv import load_dotenv
import re

load_dotenv()

True

In [3]:
# Agent 1 — gpt-4o (AzureOpenAI client)
azure_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2025-01-01-preview"
)

In [4]:
# Agents 2 & 3 — Grok and Llama (OpenAI client with Azure base URL)
alt_client = OpenAI(
    base_url="https://saisirichittineni-7443-resource.services.ai.azure.com/openai/v1",
    api_key=os.getenv("AZURE_OPENAI_KEY")
)

In [5]:
print("Clients set up!")

Clients set up!


In [6]:
df = pd.read_csv("data/Reddit_askdocs_2k.csv")
df["question_text"] = df["title"].fillna("").astype(str) + " " + df["selftext"].fillna("").astype(str)
sample_1k_df = df.sample(n=1000, random_state=42).reset_index(drop=True)
print(f"Total questions: {len(sample_1k_df)}")

Total questions: 1000


In [7]:
import spacy
import scispacy

nlp = spacy.load("en_core_sci_sm")

def extract_medical_entities(text):
    doc = nlp(text)
    entities = [ent.text for ent in doc.ents if len(ent.text) > 2]
    return entities

print("scispaCy loaded!")

/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


scispaCy loaded!


/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [8]:
def get_relationship_pairs(question, entities):
    if not entities:
        return []
    prompt = f"""You are a strict medical NLP assistant. Find ONLY real medical relationship pairs.

ALLOWED relationship types:
- "drug-disease": a drug treats a disease
- "disease-symptom": a disease causes a symptom  
- "symptom-disease": a symptom indicates a disease

STRICT RULES:
1. Both terms must be real medical entities (diseases, symptoms, drugs)
2. Do NOT pair the same concept twice
3. Do NOT create reverse pairs
4. No vague terms, procedures, or body parts alone

Question: {question[:400]}
Entities: {entities}

Return ONLY valid JSON, no markdown:
{{"pairs": [["term1", "term2", "relationship_type"], ...]}}"""

    response = azure_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=400
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()
    result = json.loads(raw)
    return result.get("pairs", [])

In [9]:
results = []
for i, row in sample_1k_df.iterrows():
    text = row["question_text"]
    entities = extract_medical_entities(text)
    try:
        pairs = get_relationship_pairs(text, entities)
    except Exception as e:
        pairs = []
        print(f"Row {i} error: {e}")
    results.append({"question_text": text, "relationship_pairs": pairs})
    print(f"Q{i+1} done — {len(pairs)} pairs")
    time.sleep(0.3)

results_df = pd.DataFrame(results)
print("\nresults_df ready!")

Q1 done — 0 pairs
Q2 done — 1 pairs
Q3 done — 0 pairs
Q4 done — 1 pairs
Q5 done — 1 pairs
Q6 done — 11 pairs
Q7 done — 1 pairs
Q8 done — 2 pairs
Q9 done — 0 pairs
Q10 done — 3 pairs
Q11 done — 2 pairs
Q12 done — 6 pairs
Q13 done — 7 pairs
Q14 done — 3 pairs
Q15 done — 7 pairs
Q16 done — 1 pairs
Q17 done — 0 pairs
Q18 done — 5 pairs
Q19 done — 8 pairs
Q20 done — 1 pairs
Q21 done — 0 pairs
Q22 done — 1 pairs
Q23 done — 1 pairs
Q24 done — 0 pairs
Q25 done — 0 pairs
Q26 done — 3 pairs
Q27 done — 3 pairs
Q28 done — 0 pairs
Q29 done — 1 pairs
Q30 done — 0 pairs
Q31 done — 1 pairs
Q32 done — 3 pairs
Q33 done — 1 pairs
Q34 done — 1 pairs
Q35 done — 0 pairs
Q36 done — 0 pairs
Q37 done — 7 pairs
Q38 done — 1 pairs
Q39 done — 1 pairs
Q40 done — 1 pairs
Q41 done — 0 pairs
Q42 done — 1 pairs
Q43 done — 2 pairs
Q44 done — 0 pairs
Q45 done — 5 pairs
Q46 done — 1 pairs
Q47 done — 3 pairs
Q48 done — 0 pairs
Q49 done — 6 pairs
Q50 done — 0 pairs
Q51 done — 2 pairs
Q52 done — 20 pairs
Q53 done — 1 pairs


In [10]:
for i, row in results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {row['question_text'][:150]}")
    print(f"Pairs: {row['relationship_pairs']}")


Q1: I used to be malnourished for 1.5 years as an early teen. Now that I'm eating well, will I grow some more or will I permanently stay at this height? I
Pairs: []

Q2: My dad (70M) was informed that he needs to start dialysis. Anything we can do? My dad who is (6feet tall and about 260lbs) came home crying that his d
Pairs: [['diabetes', 'dialysis', 'disease-symptom']]

Q3: A question about anesthesia  When getting a planned procedure, you're told to fast the night before as to not risk aspiration. Done that twice, once w
Pairs: []

Q4: Hi! I (26M) cut my finger and lost feeling in the tip, what are the chances of the nerve repairing itself? 
My finger got hit by a crazy sharp knife t
Pairs: [['nerve', 'lost feeling', 'disease-symptom']]

Q5: Is it crazy to stay in a hotel to avoid family members with covid? I m27 am vaccinated. Both of my parents who I live with (also vaccinated) have covi
Pairs: [['covid', 'positive', 'disease-symptom']]

Q6: pulsating in my right collarbone, conc

In [11]:
# Cost per 1M tokens
import os
COSTS = {
    "gpt-4o":                  {"input": 2.50,  "output": 10.00},
    "grok-4-1-fast-reasoning": {"input": 3.00,  "output": 15.00},
    "Llama-3.3-70B-Instruct":  {"input": 0.23,  "output": 0.40}
}

# Reset token usage
token_usage = {
    "gpt-4o":                  {"input": 0, "output": 0},
    "grok-4-1-fast-reasoning": {"input": 0, "output": 0},
    "Llama-3.3-70B-Instruct":  {"input": 0, "output": 0}
}

def judge_pair(question, entity1, entity2, rel_type, model_name, client):
    prompt = f"""You are a medical expert validating a relationship pair extracted from a patient question.
Question: {question}
Pair to validate:
- Entity 1: {entity1}
- Entity 2: {entity2}
- Relationship type: {rel_type}
Respond with ONLY:
SCORE: 0 or 1  (1=valid, 0=invalid)
REASON: one sentence explanation"""
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    token_usage[model_name]["input"]  += response.usage.prompt_tokens
    token_usage[model_name]["output"] += response.usage.completion_tokens
    raw = response.choices[0].message.content.strip()
    score, reason = 0, "Could not parse"
    for line in raw.split("\n"):
        if line.startswith("SCORE:"):
            try:
                score = int(line.replace("SCORE:", "").strip())
            except:
                score = 0
        if line.startswith("REASON:"):
            reason = line.replace("REASON:", "").strip()
    return score, reason

In [12]:
BATCH_SIZE = 200
OUTPUT_CSV = "data/committee_results.csv"

# Remove old CSV if exists to start fresh
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)
    print("Old CSV removed, starting fresh!")

all_results_df = results_df.copy()
print(f"Using existing pairs for {len(all_results_df)} questions!")

Using existing pairs for 1000 questions!


In [13]:
# Now run committee in batches of 200
for batch_num in range(0, 1000, BATCH_SIZE):
    batch_df = all_results_df.iloc[batch_num : batch_num + BATCH_SIZE]
    print(f"\n{'='*60}")
    print(f"BATCH {batch_num//BATCH_SIZE + 1}: questions {batch_num+1} to {batch_num+len(batch_df)}")
    print(f"{'='*60}")

    committee_rows = []
    for i, row in batch_df.iterrows():
        question = row["question_text"]
        pairs = row["relationship_pairs"]
        if not pairs:
            continue
        for pair in pairs:
            if len(pair) != 3:
                continue
            entity1, entity2, rel_type = pair[0], pair[1], pair[2]
            try:
                score1, reason1 = judge_pair(question, entity1, entity2, rel_type, "gpt-4o", azure_client)
                time.sleep(0.2)
                score2, reason2 = judge_pair(question, entity1, entity2, rel_type, "grok-4-1-fast-reasoning", alt_client)
                time.sleep(0.2)
                score3, reason3 = judge_pair(question, entity1, entity2, rel_type, "Llama-3.3-70B-Instruct", alt_client)
                time.sleep(0.2)
            except Exception as e:
                print(f"  Judge error on pair {pair}: {e}")
                continue
            total = score1 + score2 + score3
            agreement_score = f"{total}/3"
            committee_label = "VALID" if total >= 2 else "INVALID"
            for agent, score, reason in [
                ("gpt-4o", score1, reason1),
                ("grok-4-1-fast-reasoning", score2, reason2),
                ("Llama-3.3-70B-Instruct", score3, reason3)
            ]:
                committee_rows.append({
                    "question": question,
                    "entity1": entity1,
                    "entity2": entity2,
                    "relationship_type": rel_type,
                    "agent": agent,
                    "score": score,
                    "reason": reason,
                    "agreement_score": agreement_score,
                    "committee_label": committee_label
                })
        time.sleep(0.3)

    # Append batch to CSV
    batch_committee_df = pd.DataFrame(committee_rows)
    if os.path.exists(OUTPUT_CSV):
        batch_committee_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
    else:
        batch_committee_df.to_csv(OUTPUT_CSV, mode="w", header=True, index=False)
    print(f"  ✅ Batch {batch_num//BATCH_SIZE + 1} done — {len(batch_committee_df)} rows saved to CSV")

    total_cost = sum(
        (token_usage[m]["input"] / 1_000_000) * COSTS[m]["input"] +
        (token_usage[m]["output"] / 1_000_000) * COSTS[m]["output"]
        for m in token_usage
    )
    print(f"  💰 Cost so far: ${total_cost:.4f}")

print("\n✅ ALL BATCHES COMPLETE!")
print(f"Results saved to {OUTPUT_CSV}")


BATCH 1: questions 1 to 200
  Judge error on pair ['constipated', 'butthole feels', 'disease-symptom']: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': True, 'severity': 'high'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  Judge error on pair ['asbestos exposure', 'cancer', 'disease-symptom']: Error code: 400 - {'id': 'chatcmpl-d145e00cd9b849c2be5d4ec4f581a', 'model': '', 'choices': [{'ind

In [15]:
# Step 3 — Append batch to CSV
batch_committee_df = pd.DataFrame(committee_rows)
if os.path.exists(OUTPUT_CSV):
    batch_committee_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
else:
    batch_committee_df.to_csv(OUTPUT_CSV, mode="w", header=True, index=False)

print(f"  ✅ Batch {batch_num//BATCH_SIZE + 1} done — {len(batch_committee_df)} rows saved to CSV")

# Cost so far after each batch
total_cost = sum(
    (token_usage[m]["input"] / 1_000_000) * COSTS[m]["input"] +
    (token_usage[m]["output"] / 1_000_000) * COSTS[m]["output"]
    for m in token_usage
)
print(f"  💰 Cost so far: ${total_cost:.4f}")

print("\n✅ ALL BATCHES COMPLETE!")
print(f"Results saved to {OUTPUT_CSV}")

  ✅ Batch 5 done — 1578 rows saved to CSV
  💰 Cost so far: $7.8361

✅ ALL BATCHES COMPLETE!
Results saved to data/committee_results.csv


In [17]:
for i, row in batch_committee_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Question: {row['question'][:150]}")
    print(f"Pair: ({row['entity1']}, {row['entity2']}, {row['relationship_type']})")
    print(f"  Agent:     {row['agent']}")
    print(f"  Score:     {row['score']}")
    print(f"  Reason:    {row['reason']}")
    print(f"  Agreement: {row['agreement_score']} → {row['committee_label']}")


Question: Low LV ESV and EDV? Hi everyone,

I am preparing for appointments with some specialists for some cardiac-type issues I have had over the past 6-7 year
Pair: (edema, cardiac-type issues, symptom-disease)
  Agent:     gpt-4o
  Score:     1
  Reason:    Edema is a recognized symptom that can be associated with various cardiac-type issues, such as heart failure or other cardiovascular conditions.
  Agreement: 3/3 → VALID

Question: Low LV ESV and EDV? Hi everyone,

I am preparing for appointments with some specialists for some cardiac-type issues I have had over the past 6-7 year
Pair: (edema, cardiac-type issues, symptom-disease)
  Agent:     grok-4-1-fast-reasoning
  Score:     1
  Reason:    Edema is a common symptom associated with various cardiac issues, such as heart failure, as explicitly listed by the patient under their cardiac-type issues.
  Agreement: 3/3 → VALID

Question: Low LV ESV and EDV? Hi everyone,

I am preparing for appointments with some specialists for som